In [72]:
import numpy as np
import os
import cohere
from dotenv import load_dotenv

In [73]:
load_dotenv()

True

In [74]:
client = cohere.Client(api_key=os.getenv("COHERE_API_KEY"))

In [75]:
sentence = "The quick brown fox jumps over the lazy dog."

In [76]:
response = client.embed(
    model="embed-english-v3.0", #embeding model
    texts=[sentence], #input text
    input_type="search_document", #type of input text
    embedding_types=["float"] #type of embedding to be generated
)

In [77]:
embedding = np.array(response.embeddings.float) 

vector embeddings are mathematical representation of text passed, in our case it is a 1 x 1024 dimensional vector, 1024 columns are sort of features of the text passed

In [78]:
print(embedding.shape)
print(embedding[0])

(1, 1024)
[ 0.01216888 -0.02250671  0.02551269 ...  0.01032257 -0.04595947
 -0.00075817]


these embeddings are numbers difficult to understand but the main application of it to check similarity between two pieces of text

consine similarity is one major way to compute the similarity between to vector embeddings

it takes to vectors normalise them to one dimensional vector, take their dot product and the final number become the similarity score of two vectors

similarity score ranges between 1 to 0

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
# consine similarity requires 2D array as input, so we need to reshape the embedding to 2D array

In [80]:
input_1 = "exercise is good for health"
input_2 = "healthy lifestyle includes regular exercise"
input_3 = "Ram studied in the library for his exams"

In [81]:
get_embedding = lambda text: np.array(client.embed(
    model="embed-english-v3.0",
    texts=text,
    input_type="search_document",
    embedding_types=["float"]
).embeddings.float)

In [82]:
input_1_embedding = get_embedding([input_1])
input_2_embedding = get_embedding([input_2])
input_3_embedding = get_embedding([input_3])

In [83]:
similarity_1_2 = cosine_similarity(input_1_embedding, input_2_embedding)
similarity_1_3 = cosine_similarity(input_1_embedding, input_3_embedding)
similarity_2_3 = cosine_similarity(input_2_embedding, input_3_embedding)

In [84]:
print(F"similarity_1_2: {similarity_1_2[0][0]}")
print(F"similarity_1_3: {similarity_1_3[0][0]}")
print(F"similarity_2_3: {similarity_2_3[0][0]}")

similarity_1_2: 0.7838669963283964
similarity_1_3: 0.2634479364830082
similarity_2_3: 0.2637415397626617


here you can see the similarity between input 1 and input 2 is maximum 0.78 but any other combination have a similarity score lesser then 0.5

why sentance embedding is more powerfull then word embeddings?

in following example we take two sentence input 1 and input 2, remove the stoping words like is, the, in, was which have less semantic/contextual meaning in a sentence. Then compute the embedding for each word separately and final took mean of every words embedding to compute the final embedding for the sentence and final took cosine similarity of both the final vector embedding which comes to be extremely high even though the meaning of input 1 and input 2 is completely different, they aren't contextually similar. Neither they have any sense of words ordering in sentence.

when you compute the embedding of entire sentence it capture the semantic/context of the sentance as a result representing a more accurate contextual embedding

In [89]:
input_1 = "kids play in the park"
input_2 = "the play is for kids in the park"

In [90]:
input_1_embedding = get_embedding([input_1])
input_2_embedding = get_embedding([input_2])

print(cosine_similarity(input_1_embedding, input_2_embedding)[0][0])

0.880428372482752


In [91]:
input_1 = ["kids", "play", "park"]
input_2 = ["play", "kids", "park"]

In [92]:
input_1_embedding = get_embedding(input_1)
input_2_embedding = get_embedding(input_2)

In [93]:
input_1_mean_embedding = input_1_embedding.mean(axis=0)
input_2_mean_embedding = input_2_embedding.mean(axis=0)


In [94]:
similarity_1_2 = cosine_similarity([input_1_mean_embedding], [input_2_mean_embedding])


In [95]:
print(f"cosine similarity between input_1 and input_2: {similarity_1_2[0][0]}")

cosine similarity between input_1 and input_2: 1.0
